# Dataset Creation

End-to-end pipeline from `.raw` files on disk to a cached patch dataset ready for
U-Net training. The pipeline converts raw files to calibrated zarr stores (via echopype),
runs ensemble bottom detection to generate seabed masks, and extracts overlapping
3-channel patches (38 kHz, 200 kHz, frequency difference).

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import convert_raw_dir, load_zarr, list_channels
from deep_learning.echo_dataset import save_patches_cached
from deep_learning.training.config import PipelineConfig

## Configuration

All paths and extraction parameters in one place. The raw files are spread across
nine download folders under `Datasets/raw_files/`. Zarr stores go into a shared
output directory, and extracted patches are cached as `.npz` files so re-runs skip
the expensive bottom detection step.

In [ ]:
DATASETS = REPO_ROOT / 'Datasets'
RAW_ROOT = DATASETS / 'raw_files'
ZARR_OUT = DATASETS / 'echopype_zarr'       # all converted zarr stores go here
PATCH_CACHE = DATASETS / 'patch_cache'      # cached .npz patches

# Raw subdirectories (each contains .raw files from a single download batch)
RAW_DIRS = sorted([
    d for d in RAW_ROOT.iterdir()
    if d.is_dir() and d.name.startswith('drive-download')
])

cfg = PipelineConfig(
    freq1_hz=38_000,
    freq2_hz=200_000,
    patch_width=256,
    stride=64,
    vmin=-80.0,
    vmax=-34.0,
    cache_dir=PATCH_CACHE,
)

print(f'{len(RAW_DIRS)} raw directories found')

## Discover raw files

In [ ]:
all_raw = []
for d in RAW_DIRS:
    files = sorted(d.glob('*.raw'))
    all_raw.extend(files)
    print(f'  {d.name}: {len(files)} files')

total_gb = sum(f.stat().st_size for f in all_raw) / 1e9
print(f'\nTotal: {len(all_raw)} raw files ({total_gb:.1f} GB)')

## Convert raw to zarr

Each `.raw` file is converted to a calibrated zarr store via echopype. The conversion
computes Sv from the raw complex samples using the transducer and environment parameters
embedded in the file. Existing zarr stores are skipped automatically.

In [ ]:
ZARR_OUT.mkdir(parents=True, exist_ok=True)
all_zarr = []
for d in RAW_DIRS:
    zarr_paths = convert_raw_dir(d, output_dir=ZARR_OUT, skip_existing=True)
    all_zarr.extend(zarr_paths)

print(f'\n{len(all_zarr)} zarr stores in {ZARR_OUT}')

## Verify zarr stores

Quick check that both frequencies (38 kHz and 200 kHz) are available in each zarr
and that the data dimensions are reasonable.

In [ ]:
usable_zarr = []
skipped = []

for zp in sorted(all_zarr):
    try:
        channels = list_channels(zp)
        freqs_khz = sorted(set(int(ch.get('frequency_hz', 0)) // 1000 for ch in channels))
        has_38 = any(abs(f - 38) <= 1 for f in freqs_khz)
        has_200 = any(abs(f - 200) <= 2 for f in freqs_khz)

        if has_38 and has_200:
            data = load_zarr(zp, frequency=cfg.freq1_hz)
            print(f'  ✓ {zp.name:>30s}  {data.n_pings:>5d} pings  '
                  f'{data.n_samples:>5d} samples  freqs: {freqs_khz} kHz')
            usable_zarr.append(zp)
        else:
            print(f'  ✗ {zp.name:>30s}  missing frequency  (has: {freqs_khz} kHz)')
            skipped.append(zp)
    except Exception as e:
        print(f'  ✗ {zp.name:>30s}  error: {e}')
        skipped.append(zp)

print(f'\n{len(usable_zarr)} usable zarr stores, {len(skipped)} skipped')

## Extract patches

For each zarr store, the extraction pipeline normalises both frequencies to [0, 1],
runs ensemble bottom detection (simple + edge detectors) to build a binary seabed mask,
and slides a 256-ping window along the file. Patches are cached as `.npz` files so
subsequent runs load instantly.

In [ ]:
PATCH_CACHE.mkdir(parents=True, exist_ok=True)

file_stats = []  # (name, n_patches)

for i, zp in enumerate(usable_zarr):
    print(f'[{i+1}/{len(usable_zarr)}] {zp.name}')
    patches, masks = save_patches_cached(
        zp, PATCH_CACHE,
        freq1_hz=cfg.freq1_hz,
        freq2_hz=cfg.freq2_hz,
        patch_width=cfg.patch_width,
        stride=cfg.stride,
        vmin=cfg.vmin,
        vmax=cfg.vmax,
        require_bottom=True,
    )
    file_stats.append((zp.name, len(patches)))
    print(f'    → {len(patches)} patches')

total_patches = sum(n for _, n in file_stats)
print(f'\nTotal: {total_patches} patches from {len(file_stats)} files')

## Dataset statistics

In [ ]:
counts = [n for _, n in file_stats]

print(f'Files:            {len(file_stats)}')
print(f'Total patches:    {total_patches}')
print(f'Patches per file: {np.mean(counts):.1f} ± {np.std(counts):.1f}  '
      f'(min {np.min(counts)}, max {np.max(counts)})')

fig, ax = plt.subplots(figsize=(12, 4))
names = [name.replace('.zarr', '') for name, _ in file_stats]
ax.bar(range(len(counts)), counts, color='steelblue')
ax.set_xlabel('File')
ax.set_ylabel('Patches')
ax.set_title(f'Patches per file ({total_patches} total)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=90, fontsize=6)
plt.tight_layout()
plt.show()

# Class balance: load a few cached patches and check bottom pixel fraction
sample_npz = sorted(PATCH_CACHE.glob('*.npz'))[:5]
bottom_fracs = []
for npz_path in sample_npz:
    with np.load(npz_path) as data:
        for m in data['masks']:
            bottom_fracs.append(m.sum() / m.size)

print(f'\nBottom pixel fraction (sampled from {len(bottom_fracs)} patches):')
print(f'  mean={np.mean(bottom_fracs):.3f}  std={np.std(bottom_fracs):.3f}  '
      f'min={np.min(bottom_fracs):.3f}  max={np.max(bottom_fracs):.3f}')

## Sanity check

A few random patches with their masks overlaid. Each row shows channel 0 (38 kHz),
channel 1 (200 kHz), and the binary seabed mask.

In [ ]:
# Load a random sample of patches for visualization
rng = np.random.RandomState(42)
all_npz = sorted(PATCH_CACHE.glob('*.npz'))
sample_file = rng.choice(all_npz)
with np.load(sample_file) as data:
    sample_patches = data['patches']
    sample_masks = data['masks']

n_show = min(6, len(sample_patches))
indices = rng.choice(len(sample_patches), size=n_show, replace=False)

fig, axes = plt.subplots(n_show, 3, figsize=(14, 3 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    img = sample_patches[idx]   # (3, H, W)
    msk = sample_masks[idx]     # (H, W)

    axes[row, 0].imshow(img[0], aspect='auto', cmap='viridis', vmin=0, vmax=1)
    axes[row, 0].set_ylabel(f'Patch {idx}')
    if row == 0:
        axes[row, 0].set_title('38 kHz')

    axes[row, 1].imshow(img[1], aspect='auto', cmap='viridis', vmin=0, vmax=1)
    if row == 0:
        axes[row, 1].set_title('200 kHz')

    axes[row, 2].imshow(msk, aspect='auto', cmap='gray_r', vmin=0, vmax=1)
    if row == 0:
        axes[row, 2].set_title('Seabed mask')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f'Sample patches from {sample_file.name}', fontsize=11)
plt.tight_layout()
plt.show()

## Summary

The dataset is ready. The patch cache directory and usable zarr list are the inputs
for notebook 16 (training). To re-run with different parameters (patch size, stride,
frequency range), change the config above and clear the cache directory — the zarr
conversion step will still be skipped.

In [ ]:
print(f'Zarr stores:  {ZARR_OUT}')
print(f'Patch cache:  {PATCH_CACHE}')
print(f'Usable files: {len(usable_zarr)}')
print(f'Total patches: {total_patches}')
print(f'Patch shape:  (3, H, {cfg.patch_width})  — 38 kHz / 200 kHz / difference')
print(f'Stride:       {cfg.stride} pings')
print(f'Sv range:     [{cfg.vmin}, {cfg.vmax}] dB')